In [1]:
import os
import pandas as pd
from Bio import Phylo
from pathlib import Path
if not Path('jw_utils').exists():
    !git clone https://github.com/JonWinkelman/jw_utils.git
from jw_utils import itol_annotations as ia
from jw_utils import ncbi_utils as nu
from jw_utils import parse_fasta as pfa
if not Path('orthofinder_utils').exists():
    !git clone https://github.com/JonWinkelman/orthofinder_utils.git
from orthofinder_utils import dash_ortho_parser as dop
from orthofinder_utils import run_orthofinder as ru
import external_functions as ef

### Download proteomes and GFF files from NCBI datasets

In [2]:
results_dir = Path('./results/4_orthofinder_raxML')
results_dir.mkdir(exist_ok=True)

In [3]:
figure_genome_assemblies = {
    'GCF_001077675.1': 'Acinetobacter baumannii ATCC 17978-mff',
    'GCF_000018445.1': 'Acinetobacter baumannii ACICU',
    'GCF_000021245.2': 'Acinetobacter baumannii AB0057',
    'GCF_000770605.1': 'Acinetobacter baumannii AB5075',
    'GCF_005281455.1': 'Acinetobacter nosocomialis M2',
    'GCF_002055515.1': 'Acinetobacter calcoaceticus CA16',
    'GCF_001682515.1': 'Acinetobacter gyllenbergii FMP01',
    'GCF_000413935.1': 'Acinetobacter colistiniresistens NIPH 2036',
    'GCF_000046845.1': 'Acinetobacter baylyi ADP1',
    'GCF_000368805.1': 'Acinetobacter johnsonii ANC 3681',
    'GCF_004208515.1': 'Acinetobacter halotolerans JCM 31009',
    'GCF_001105265.1': 'Yersinia enterocolitica SC9312-78',
    }
figure_accs = list(figure_genome_assemblies.keys())


#### Download assembly summaries

temp_file created at /var/folders/vw/7lg51dfd3ql9g_j55xz_3dqh0000gn/T/tmpkzcylq7y


New version of client (18.21.0) available at https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/mac/datasets.


## Download proteomes/gffs and run orthofinder 2.5.5 on proteomes

In [5]:
nu.download_genomes_from_acclist(figure_accs)
orthofinder_base_dir   = results_dir / 'orthofinder_analysis'
proteomes_dir = orthofinder_base_dir / 'data/proteomes'
gff_dir       = orthofinder_base_dir / 'data/gffs'
proteomes_dir.mkdir(exist_ok=True, parents=True)
gff_dir.mkdir(exist_ok=True, parents=True)
nu.copy_ncbi_files('./ncbi_dataset/ncbi_dataset/data/', dest_dir=proteomes_dir, suffix='.faa')
nu.copy_ncbi_files('./ncbi_dataset/ncbi_dataset/data/', dest_dir=gff_dir, suffix='.gff')

temp_file created at /var/folders/vw/7lg51dfd3ql9g_j55xz_3dqh0000gn/T/tmpucobruwi
Error: New version of client (18.22.1) available at https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/mac/datasets.
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 6.08MB/s
Downloading: ncbi_dataset.zip    847B 

In [6]:
append = "Acinetobacter"
orthofinder_output_dir = orthofinder_base_dir / "data/orthofinder_results"
#ru.run_orthofinder(str(proteomes_dir), o=orthofinder_output_dir, n=append)
orthofinder_output_dir = orthofinder_output_dir / f'Results_{append}'

### Parse orthofinder output to find HOGs. 

In [7]:
#make summary files
dop_obj = dop.DashOrthoParser(orthofinder_output_dir)

summary_dir = orthofinder_output_dir / 'summary_data'
summary_dir.mkdir(exist_ok=True)
summary_file = summary_dir / 'summaries.json'
nu.download_assembly_summaries_from_list(figure_accs, output_file=summary_file)
summary_df = nu.make_summary_df(summary_file)
summary_df['organism_name'].to_json(summary_dir / 'AssemblyAccession_to_SpeciesName.json')

temp_file created at /var/folders/vw/7lg51dfd3ql9g_j55xz_3dqh0000gn/T/tmppgb0eypq


New version of client (18.22.1) available at https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/mac/datasets.


In [8]:
genome_annotation_df = dop_obj.make_genome_annotation_df('GCF_001077675.1',get_common_names=True)
astA = 'gene-ACX60_RS01825'

astR = 'gene-ACX60_RS12905'
astN = 'gene-ACX60_RS12900'
astO = 'gene-ACX60_RS12895'
astP = 'gene-ACX60_RS12890'

astR_HOG = genome_annotation_df.set_index('Parents').loc['gene-ACX60_RS12905', 'HOGs']
astR_orthologs = dop_obj.all_prots_in_HOG(astR_HOG)

astR_accs = [a[:15][::-1].replace('_', '.', 1)[::-1] for a in astR_orthologs]
astR_proteins = dop_obj.get_HOG_protein_seqs(astR_HOG)
astR_seqs_d = astR_proteins['Protein_seq'].to_dict()

PosixPath('results/4_orthofinder_raxML')

### Align with muscle and Run RAxML

In [9]:
tree_dir = Path(results_dir / 'raxML_tree')
tree_dir.mkdir(exist_ok=True)
astR_seqs_fp = tree_dir / 'astR_orthologs.faa'
pfa.write_to_fasta(astR_seqs_d, astR_seqs_fp)

In [11]:
muscle_out= str(astR_seqs_fp).replace('.faa', '.muscle.aln')
ef.run_muscle(astR_seqs_d, muscle_out)

Running: muscle -in /var/folders/vw/7lg51dfd3ql9g_j55xz_3dqh0000gn/T/muscle_rgvjxsl6.fa -out results/4_orthofinder_raxML/raxML_tree/astR_orthologs.muscle.aln
MUSCLE v3.8.1551 by Robert C. Edgar

http://www.drive5.com/muscle
This software is donated to the public domain.
Please cite: Edgar, R.C. Nucleic Acids Res 32(5), 1792-97.

muscle_rgvjxsl6 12 seqs, lengths min 140, max 153, avg 141
00:00:00      2 MB(0%)  Iter   1    1.28%  K-mer dist pass 1
00:00:00      2 MB(0%)  Iter   1  100.00%  K-mer dist pass 1
00:00:00      2 MB(0%)  Iter   1    1.28%  K-mer dist pass 2
00:00:00      2 MB(0%)  Iter   1  100.00%  K-mer dist pass 2
00:00:00      3 MB(0%)  Iter   1    9.09%  Align node       
00:00:00      3 MB(0%)  Iter   1   18.18%  Align node
00:00:00      3 MB(0%)  Iter   1   27.27%  Align node
00:00:00      3 MB(0%)  Iter   1   36.36%  Align node
00:00:00      3 MB(0%)  Iter   1   45.45%  Align node
00:00:00      3 MB(0%)  Iter   1   54.55%  Align node
00:00:00      4 MB(0%)  Iter   1   

'results/4_orthofinder_raxML/raxML_tree/astR_orthologs.muscle.aln'

In [12]:
raxml_out_dir = tree_dir / 'raxML_output'
ef.run_raxml(muscle_out, raxml_out_dir, prefix='AstR', threads=8, model='LG', raxml_bin='raxmlHPC')

In [14]:
itol_annotations_dir = tree_dir / 'itol_annotations'
itol_annotations_dir.mkdir(exist_ok=True)
tree = Phylo.read(file=f'{raxml_out_dir}/RAxML_bipartitionsBranchLabels.AstR', format='newick')
relable_d = {}
for cl in tree.get_terminals():
    acc = cl.name[:15][::-1].replace('_', '.', 1)[::-1]
    relable_d[cl.name] = dop_obj.accession_to_name[acc]
itol_relable_out = itol_annotations_dir / 'RELABLE_RAxML_bipartitionsBranchLabels.astR'
ia.relabel_itol_treeleafs(relable_d, itol_relable_out )

## Make species tree 

In [15]:
from jw_utils import make_bac120_tree as mbt
hmm_profile_dir = results_dir / 'bac120_hmm_profiles'
proteome_dir_fp = results_dir / 'orthofinder_analysis/data/orthofinder_results/Results_Acinetobacter/proteomes/'
output_tree     = results_dir / 'bac120_tree/bac120_tree.nwk'
#mbt.make_bac120_tree(str(hmm_profile_dir), str(proteome_dir_fp),output_tree = str(output_tree))

In [16]:
from jw_utils import  itol_annotations as ia
itol_annotations_dir = results_dir / 'itol_annotations'
itol_annotations_dir.mkdir(exist_ok=True)

In [64]:
ia.relabel_itol_treeleafs(summary_df['organism_name'].to_dict(), itol_annotations_dir / 'RELABLE_bac120_tree.txt')